# Spam Text Classification Training (Pro)
This notebook trains a high-accuracy binary classifier to detect spam messages using real-world data:
1. **SMS Spam Collection** (`spam.csv`) -> Label 1 (Spam), Label 0 (Ham)

In [1]:
import pandas as pd
import pickle
import os
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Setup NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [2]:
def transform_text(text):
    text = str(text).lower()
    text = nltk.word_tokenize(text)
    y = [i for i in text if i.isalnum()]
    text = [ps.stem(i) for i in y if i not in stop_words and i not in punctuation]
    return " ".join(text)

## 1. Load and Clean Data

In [3]:
print("Loading dataset...")
df = pd.read_csv('spam.csv', encoding='latin1')

# Drop unnecessary columns
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})

# Encode labels: ham -> 0, spam -> 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df.dropna(subset=['text'], inplace=True)
df.drop_duplicates(inplace=True)

print(f"Total samples: {len(df)}")
print(df['label'].value_counts())

Loading dataset...
Total samples: 5169
label
0    4516
1     653
Name: count, dtype: int64


In [4]:
print("Processing text...")
df['transformed_text'] = df['text'].astype(str).apply(transform_text)

print("Vectorizing and Training...")
tfidf = TfidfVectorizer(max_features=30000)
X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['label'].values

model = MultinomialNB()
model.fit(X, y)

with open('vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('model.pkl', 'wb') as f: pickle.dump(model, f)
print("Retraining complete with MultinomialNB!")

Processing text...
Vectorizing and Training...
Retraining complete with MultinomialNB!
